<a href="https://colab.research.google.com/github/ErickReyes894/Ejercicio-final-/blob/main/%20Cuello%20de%20Botella%20a%20la%20Concurrencia%20Multihilo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# CELDA 1 — Servidor secuencial (ejecutar primero)
import socket
import threading
import time

def iniciar_servidor_secuencial():
    HOST, PORT = '127.0.0.1', 8000
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind((HOST, PORT))
        s.listen(5)
        print("[Servidor] Listo en 127.0.0.1:8000 — modo SECUENCIAL")
        print("[Servidor] Esperando conexiones...\n")

        for _ in range(2):  # atiende exactamente 2 clientes y termina
            conn, addr = s.accept()
            print(f"[Servidor] Conexión aceptada de {addr}")
            print(f"[Servidor] Procesando... (pausa de 10s — nadie más puede conectar)")
            with conn:
                time.sleep(10)          # EL CUELLO DE BOTELLA
                conn.sendall(b"[Servidor] Listo! (modo secuencial)\n")
            print(f"[Servidor] Conexión cerrada con {addr}\n")

t = threading.Thread(target=iniciar_servidor_secuencial, daemon=True)
t.start()
time.sleep(0.5)
print(">>> Servidor corriendo en background. Ejecuta la CELDA 2 ahora.\n")

[Servidor] Listo en 127.0.0.1:8000 — modo SECUENCIAL
[Servidor] Esperando conexiones...

>>> Servidor corriendo en background. Ejecuta la CELDA 2 ahora.



In [3]:
# CELDA 2 — Dos clientes simultáneos (ejecutar después de la celda 1)
import socket
import threading
import time

def cliente(nombre, retraso_inicial=0):
    time.sleep(retraso_inicial)
    t0 = time.time()
    print(f"[{nombre}] Intentando conectar en t={retraso_inicial}s...")

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.connect(('127.0.0.1', 8000))
        print(f"[{nombre}] Conectado. Esperando respuesta del servidor...")
        respuesta = s.recv(1024).decode().strip()
        duracion = time.time() - t0
        print(f"[{nombre}] Respuesta recibida en {duracion:.1f}s --> {respuesta}")

t1 = threading.Thread(target=cliente, args=("Cliente-1", 0))
t2 = threading.Thread(target=cliente, args=("Cliente-2", 1))

t1.start()
t2.start()

t1.join()
t2.join()

print("\n[Analisis]")
print("  Cliente-1 respondido en ~10s  (lo atendieron de inmediato)")
print("  Cliente-2 respondido en ~20s  (tuvo que esperar que terminara Cliente-1)")
print("  --> El servidor BLOQUEÓ a Cliente-2 durante 10 segundos extra.")

[Cliente-1] Intentando conectar en t=0s...
[Cliente-1] Conectado. Esperando respuesta del servidor...
[Servidor] Conexión aceptada de ('127.0.0.1', 49900)
[Servidor] Procesando... (pausa de 10s — nadie más puede conectar)
[Cliente-2] Intentando conectar en t=1s...
[Cliente-2] Conectado. Esperando respuesta del servidor...
[Cliente-1] Respuesta recibida en 10.0s --> [Servidor] Listo! (modo secuencial)[Servidor] Conexión cerrada con ('127.0.0.1', 49900)


[Servidor] Conexión aceptada de ('127.0.0.1', 49908)
[Servidor] Procesando... (pausa de 10s — nadie más puede conectar)
[Servidor] Conexión cerrada con ('127.0.0.1', 49908)

[Cliente-2] Respuesta recibida en 19.0s --> [Servidor] Listo! (modo secuencial)

[Analisis]
  Cliente-1 respondido en ~10s  (lo atendieron de inmediato)
  Cliente-2 respondido en ~20s  (tuvo que esperar que terminara Cliente-1)
  --> El servidor BLOQUEÓ a Cliente-2 durante 10 segundos extra.
